In [0]:
/* Viewership data */
select * 
from `brighttv_analytics`.`default`.`viewership` 
limit 100;


/*User profile data*/
select *
from `brighttv_analytics`.`default`.`user_profile` 
limit 100;


/*Gathering data from the Viewership sheet*/

dA/
/*day and time of first and last day of data collection */
select min(LocalTime_SAST) as first_Collection_Day,
       MAX(LocalTime_SAST) AS last_Collection_Day
       From `brighttv_analytics`.`default`.`viewership` ;

/*months of data collection*/
select Distinct month(TO_TIMESTAMP(LocalTime_SAST,'M/d/yyyy H:mm')) as Month_Number,
        Date_format(TO_TIMESTAMP(LocalTime_SAST,'M/d/yyyy H:mm'),'MMMM') AS Month_Name
From  `brighttv_analytics`.`default`.`viewership`
order by Month_Number;

/*Number of clicked views per month*/
select Distinct month(TO_TIMESTAMP(LocalTime_SAST,'M/d/yyyy H:mm')) as Month_Number,
        Date_format(TO_TIMESTAMP(LocalTime_SAST,'M/d/yyyy H:mm'),'MMMM') AS Month_Name,
Count(*) as no_Viewers
From  `brighttv_analytics`.`default`.`viewership`
group by Month_Number, Month_Name
order by  no_Viewers desc;

/*Number of channels*/
select Count (distinct Channel2 ) as Num_Channels
from `brighttv_analytics`.`default`.`viewership`;


/* Popular channel by number of views */
select distinct Channel2 as Channel_Names,
     count (Channel2) as No_Views_Per_Channel
     from `brighttv_analytics`.`default`.`viewership`
     group by Channel_Names
     order by No_Views_Per_Channel Desc;

/*Channel categories*/
Select Channel2 as Channel_Names,
        case
        when Channel2 in('Supersport Live Events','ICC Cricket World Cup 2011','SuperSport Blitz','SuperSport Live Events','Wimbledon','Live on SuperSport') THEN 'Sports Channel'
        When Channel2 in ('Trace TV','Channel O','E! Entertainment','Vuzu','MK') then 'Music And Entertainment'
        When Channel2 in ('Africa Magic', 'M-Net', 'kykNET','Sawsee') then 'Movies, Drama & Series'
        when Channel2 in ('Cartoon Network', 'Boomerang') then 'Kids channel'
        Else 'Events & Technical'
End as Channel_Categories
from `brighttv_analytics`.`default`.`viewership`
group by Channel_Names;



/*Total Views duration */
SELECT 
    CONCAT(
        FLOOR(total_seconds / 3600), ':',
        FLOOR((total_seconds % 3600) / 60), ':',
        FLOOR(total_seconds % 60)) AS total_Viewing_duration
FROM (
select sum(hour(`Duration 2`) * 3600 +
      Minute(`Duration 2`) *60 +
      second (`Duration 2`))
      as total_Seconds
      from `brighttv_analytics`.`default`.`viewership`);


/* popular channel by total time/hours viewed*/
select Channel2 as Channel_Names,
   concat(floor(sum(hour(`Duration 2`)*3600+ minute(`Duration 2`)*60 + second(`Duration 2`))/3600),':',
          floor((sum(hour(`Duration 2`)*3600+ minute(`Duration 2`)*60 + second(`Duration 2`))/3600)/60),':',
            floor(sum(hour(`Duration 2`)*3600+ minute(`Duration 2`)*60 + second(`Duration 2`))/60)) as Total_Time
  from `brighttv_analytics`.`default`.`viewership`
  GROUP BY Channel2
ORDER BY total_time DESC;
 

/*most popular day of the week by total number of clicked views*/
SELECT date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'EEEE') AS Day_Name,
    COUNT(*) AS Views
FROM `brighttv_analytics`.`default`.`viewership`
GROUP BY date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'EEEE')
ORDER BY Views DESC;


/*Most popular time of the day by number of clicks */
SELECT date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'HH:00' ) AS hour_of_day,
       COUNT(*) AS Total_Views
FROM `brighttv_analytics`.`default`.`viewership`
GROUP BY  date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'HH:00')
ORDER BY Total_Views DESC;


/*top 10 most popular channels and days viewed by number of clicks */
SELECT date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'EEEE') AS Day_Of_Week,
       Channel2 AS Channel_Name,
       COUNT(*) AS Total_Views
FROM `brighttv_analytics`.`default`.`viewership`
GROUP BY date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'EEEE'),
       Channel2
ORDER BY Total_Views DESC
limit 10;


/*most popular channel on each day of the week*/
WITH ranked AS (SELECT date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'), 'EEEE') AS Day_Of_Week,
                       Channel2 AS Channel_Name,
                      count(*) AS Total_Views,
                      ROW_NUMBER() OVER (
                    PARTITION BY date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'), 'EEEE')
                    ORDER BY count(*) DESC) as RN
FROM `brighttv_analytics`.`default`.`viewership`
GROUP BY date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'), 'EEEE'),
        Channel2)
SELECT Day_Of_Week,
       Channel_Name,
       Total_Views
FROM ranked
where RN=1
ORDER BY Total_Views DESC;


/*longest and shortrest watch duration*/
SELECT  CONCAT(FLOOR(MIN(seconds) / 3600), ':',
        FLOOR((MIN(seconds) % 3600) / 60), ':',
        FLOOR(MIN(seconds) % 60)) AS Shortest_Duration,
     CONCAT(FLOOR(MAX(seconds) / 3600), ':',
        FLOOR((MAX(seconds) % 3600) / 60), ':',
        FLOOR(MAX(seconds) % 60)) AS Longest_Duration

FROM (  SELECT 
        HOUR(to_timestamp(`Duration 2`)) * 3600 +
        MINUTE(to_timestamp(`Duration 2`)) * 60 +
        SECOND(to_timestamp(`Duration 2`)) AS seconds
    FROM `brighttv_analytics`.`default`.`viewership`);


/*Each channels longest watch time duration*/
SELECT Channel_Name, 
     CONCAT(FLOOR(MAX(seconds) / 3600), ':',
        FLOOR((MAX(seconds) % 3600) / 60), ':',
        FLOOR(MAX(seconds) % 60)) AS Longest_Duration
FROM (  SELECT 
        Channel2 AS Channel_Name,
        HOUR(to_timestamp(`Duration 2`)) * 3600 +
        MINUTE(to_timestamp(`Duration 2`)) * 60 +
        SECOND(to_timestamp(`Duration 2`)) AS seconds
FROM `brighttv_analytics`.`default`.`viewership`)
GROUP BY Channel_Name
ORDER BY Longest_Duration DESC;

/*Channel categories*/
select Channel2 as Channel_Name,
 CASE
        WHEN Channel2 IN ('Supersport Live Events','ICC Cricket World Cup 2011','SuperSport Blitz','SuperSport Live Events','Wimbledon','Live on SuperSport')
            THEN 'Sports'
        WHEN Channel2 IN ('Trace TV','Channel O','E! Entertainment','Vuzu','MK')
            THEN 'Music & Entertainment'
        WHEN Channel2 IN ('Africa Magic','M-Net','kykNET','Sawsee')
            THEN 'Movies & Series'
        WHEN Channel2 IN ('Cartoon Network','Boomerang')
            THEN 'Kids'
        ELSE 'Events & Technical'
    END AS Channel_Category
FROM `brighttv_analytics`.`default`.`viewership`
GROUP BY Channel_Name;

select LocalTime_SAST,
    CASE
        When hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 0 AND 3 THEN 'Mid-Night to Early Morning'
        When hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 4 AND 7 THEN 'Morning'
        When hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 8 AND 11 THEN 'Late Morning'
        When hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 12 AND 15 THEN 'Afternoon'
        When hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 16 AND 19 THEN 'Evening'
        ELSE 'Night'
    END AS Time_Category
from `brighttv_analytics`.`default`.`viewership`;


/*Joined tables to get data of all clients that are active and not active*/

/*Join the tables by UserID using full outer join, it will show inactive subscribers as well*/
SELECT *
FROM `brighttv_analytics`.`default`.`user_profile`  AS A
FULL OUTER JOIN `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID;

/*Total number of subscribers incl inactive subscribers*/
select count(distinct *) as Tot_Viewers
from `brighttv_analytics`.`default`.`user_profile`  AS A
FULL OUTER JOIN `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID;


/*Number of subscribers excl inactive subscribers*/
select count(distinct *) as Tot_Viewers
from `brighttv_analytics`.`default`.`user_profile`  AS A
FULL OUTER JOIN `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID
where A.Name <> 'None' and 
        A.Surname <> 'None';


 /*Total number of inactive subscribers*/
select count(*) as Missing_Info_Subcribers
from `brighttv_analytics`.`default`.`user_profile`  AS A
full outer join `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID
where A.Name = 'None' OR A.Surname = 'None';
 

/*emails of the subscribers that did not fill in theri names, 
NB!! mail them to fill in missing data*/
select  A.UserID,
        A.Email
FROM    `brighttv_analytics`.`default`.`user_profile`  AS A
full outer join `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID 
where A.Name = 'None' OR A.Surname = 'None';
 

/*The most active subscriber*/
Select  A.UserID,
        A.Name,
        A.Surname, 
        Count(A.UserID,B.UserID) as No_Clicks
from `brighttv_analytics`.`default`.`user_profile`  AS A
Inner join `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID
group by A.UserID,A.Name,A.Surname
order by No_Clicks desc;
  

/*Most active vs least active views*/
Select  min(*) as Lowest_viewership,
        Max(*) as Highest_Views,
        Avg(*) as Avg_views
from `brighttv_analytics`.`default`.`user_profile`  AS A
Inner join `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID;
 


/*Viewership status, Active viewer, mid active viewer, not active viewer*/
Select  A.UserID,
        A.Name,
        A.Surname, 
        Count(A.UserID,B.UserID) as No_Clicks,
        Case 
                when No_Clicks >= 30 then 'Active Viewer'
                When No_Clicks between 12 and 29 then 'Mid Active Viewer'
                When No_Clicks <1
                 then 'Inactive viewer'
                else 'Not Active Viewer'
        End as Viewership_Status
from `brighttv_analytics`.`default`.`user_profile`  AS A
full outer join `brighttv_analytics`.`default`.`viewership`  AS B
ON A.UserID = B.UserID
group by A.UserID,A.Name,A.Surname
order by No_Clicks asc;
 

/*User profiles table*/
select * 
from `brighttv_analytics`.`default`.`user_profile`;


/*Age */
select distinct Age
from `brighttv_analytics`.`default`.`user_profile`
order by Age desc;


/*Youngest and Oldest viewer*/
select Min(Age) as Youngest_viewer,
        Max(Age) as Oldest_viewer
from     `brighttv_analytics`.`default`.`user_profile`;


/*count number of subscribers in each Age group*/not right heading
select DISTINCT UserID, Name, Surname,Age,
        Case 
                When Age between 0 and 9 then '0-9'
                when Age between 10 and 19 then '10-19'
                When Age between 20 and 29 then '20-29'
                When Age between 30 and 39 then '30-39'
                when Age between 40 and 49 then '40-49'
                when Age between 50 and 59 then '50-59'
                when Age between 60 and 69 then '60-69'
                when Age between 70 and 79 then '70-79'
                When Age between 80 and 89 then '80-89'
                Else '90+'
        End as Age_Group,        
        Case 
                when Age between 0 and 4 then 'Child/Infant'
                when Age between 5 and 12 then 'Children'
                when Age between 13 and 17 then 'Teenagers'
                when Age between 18 and 24 then 'Young Adults'
                When Age between 25 and 34 then 'Adult'
                when Age between 35 and 44 then 'Mid Adult'
                when Age between 45 and 54 then 'Mature Adult'
                when Age between 55 and 64 then 'Pre_Retirement'
                when Age between 65 and 74 then 'Seniors'
                When Age between 75 and 84 then 'Elderly'
                When Age >= 35 then 'Advanced Age'
        End as Age_Category
from `brighttv_analytics`.`default`.`user_profile`
group by UserID,Name,Surname,Age
order by Age desc;


/* Subscription by age group*/
select 
         Case 
                When Age between 0 and 9 then '0-9'
                when Age between 10 and 19 then '10-19'
                When Age between 20 and 29 then '20-29'
                When Age between 30 and 39 then '30-39'
                when Age between 40 and 49 then '40-49'
                when Age between 50 and 59 then '50-59'
                when Age between 60 and 69 then '60-69'
                when Age between 70 and 79 then '70-79'
                When Age between 80 and 89 then '80-89'
                Else '90+'
        End as Age_Group,        
       Count(Userid) as No_Subscribers
from `brighttv_analytics`.`default`.`user_profile`
group by Age_Group      
order by No_Subscribers desc;


/*Subscribers by age category*/
SELECT 
       CASE 
        WHEN Age BETWEEN 0 AND 12 THEN 'Child/Infant'
        WHEN Age BETWEEN 13 AND 17 THEN 'Teenagers'
        WHEN Age BETWEEN 18 AND 24 THEN 'Young Adults'
        WHEN Age BETWEEN 25 AND 34 THEN 'Adult'
        WHEN Age BETWEEN 35 AND 44 THEN 'Mid Adult'
        WHEN Age BETWEEN 45 AND 54 THEN 'Mature Adult'
        WHEN Age BETWEEN 55 AND 64 THEN 'Pre_Retirement'
        WHEN Age BETWEEN 65 AND 74 THEN 'Seniors'
        WHEN Age BETWEEN 75 AND 84 THEN 'Elderly'
        ELSE 'Advanced Age'
    END AS Age_Category,
COUNT(UserID) AS No_Subscribers
FROM `brighttv_analytics`.`default`.`user_profile`
GROUP BY Age_Category
ORDER BY No_Subscribers DESC;


/*age with the most subscribers */
select 
         Case 
                When Age between 0 and 9 then '0-9'
                when Age between 10 and 19 then '10-19'
                When Age between 20 and 29 then '20-29'
                When Age between 30 and 39 then '30-39'
                when Age between 40 and 49 then '40-49'
                when Age between 50 and 59 then '50-59'
                when Age between 60 and 69 then '60-69'
                when Age between 70 and 79 then '70-79'
                When Age between 80 and 89 then '80-89'
                Else '90+'
        End as Age_Group,        
Count(Userid) as Total_Viewers
from `brighttv_analytics`.`default`.`user_profile`
group by Age_Group      
order by Total_Viewers desc
Limit 1;


/*Total time watch by age group*/
SELECT   
        CASE
        When Age between 0 and 9 then '0-9'
        when Age between 10 and 19 then '10-19'
        When Age between 20 and 29 then '20-29'
        When Age between 30 and 39 then '30-39'
        when Age between 40 and 49 then '40-49'
        when Age between 50 and 59 then '50-59'
        when Age between 60 and 69 then '60-69'
        when Age between 70 and 79 then '70-79'
        When Age between 80 and 89 then '80-89'
        Else '90+'
    END AS Age_Group,
    CONCAT(LPAD(FLOOR(total_seconds / 3600), 2, '0'), ':',
        LPAD(FLOOR((total_seconds % 3600) / 60), 2, '0'), ':',
        LPAD(FLOOR(total_seconds % 60), 2, '0'))
    AS Total_Time_Watched

FROM (SELECT 
        A.Age,
        SUM(HOUR(B.`Duration 2`) * 3600 +
            MINUTE(B.`Duration 2`) * 60 +
            SECOND(B.`Duration 2`)) AS total_seconds
    FROM `brighttv_analytics`.`default`.`user_profile` as A
    FULL OUTER JOIN `brighttv_analytics`.`default`.`viewership` as B
    ON A.UserID = B.UserID
    GROUP BY A.Age)
    GROUP BY Age_Group, total_seconds
ORDER BY total_seconds DESC;


/*Race*/
select Distinct Race
FROM `brighttv_analytics`.`default`.`user_profile`;

/*Number of subscribers by race*/
Select Race,
        count( *) as No_of_viwers
from `brighttv_analytics`.`default`.`user_profile`
group by Race
order by No_of_viwers desc;

/*No of Subscribers by age group and rage*/
Select Race,
case
        When Age between 0 and 9 then '0-9'
        when Age between 10 and 19 then '10-19'
        When Age between 20 and 29 then '20-29'
        When Age between 30 and 39 then '30-39'
        when Age between 40 and 49 then '40-49'
        when Age between 50 and 59 then '50-59'
        when Age between 60 and 69 then '60-69'
        when Age between 70 and 79 then '70-79'
        When Age between 80 and 89 then '80-89'
        Else '90+'
    END AS Age_Group,
count (*) as No_Subscribers
from `brighttv_analytics`.`default`.`user_profile`
group by Race,Age_group
order by No_Subscribers desc;

/*Gender*/
select Gender,
        Count(*) as Female_Male
from `brighttv_analytics`.`default`.`user_profile`
group by Gender;


/*Peak active time according to age category*/ fix
SELECT   
    CASE 
        WHEN A.Age BETWEEN 0 AND 12 THEN 'Child/Infant'
        WHEN A.Age BETWEEN 13 AND 17 THEN 'Teenagers'
        WHEN A.Age BETWEEN 18 AND 24 THEN 'Young Adults'
        WHEN A.Age BETWEEN 25 AND 34 THEN 'Adult'
        WHEN A.Age BETWEEN 35 AND 44 THEN 'Mid Adult'
        WHEN A.Age BETWEEN 45 AND 54 THEN 'Mature Adult'
        WHEN A.Age BETWEEN 55 AND 64 THEN 'Pre_Retirement'
        WHEN A.Age BETWEEN 65 AND 74 THEN 'Seniors'
        WHEN A.Age BETWEEN 75 AND 84 THEN 'Elderly'
        ELSE 'Advanced Age'
    END AS Age_Category,
CONCAT(LPAD(HOUR(B.`Duration 2`), 2,'0'),' :00') AS Active_Hour,
COUNT(*) AS No_Viewers
FROM `brighttv_analytics`.`default`.`user_profile` A
INNER JOIN `brighttv_analytics`.`default`.`viewership` B
ON A.UserID = B.UserID
GROUP BY Age_Category,
    HOUR(B.`Duration 2`)
ORDER BY No_Viewers DESC;


/*The End*/


/* Combining functions to get a clean and enhanced data set*/
WITH duration_calc AS (
    SELECT 
        A.UserID,
        A.`Name`,
        A.`Surname`,
        A.`Email`,
        A.`Gender`,
        A.Age,
        A.`Race`,
        A.`Province`,
        A.`Social Media Handle` AS Instagram_Handles,
        B.`Channel2`,
        B.`LocalTime_SAST`,
        /*Counting number of views*/
        COUNT(*) AS Number_Of_Views,
        MAX(
            hour(to_timestamp(B.`Duration 2`)) * 3600 +
            minute(to_timestamp(B.`Duration 2`)) * 60 +
            second(to_timestamp(B.`Duration 2`))
        ) AS max_seconds
/*Names of tables */
    FROM `brighttv_analytics`.`default`.`viewership` B
    LEFT JOIN `brighttv_analytics`.`default`.`user_profile` A
        ON A.UserID = B.UserID
    GROUP BY 
        A.UserID,
        A.`Name`,
        A.`Surname`,
        A.`Email`,
        A.`Gender`,
        A.Age,
        A.`Race`,
        A.`Province`,
        A.`Social Media Handle`,
        B.`Channel2`,
        B.`LocalTime_SAST`)
SELECT 
    UserID,
    Name,
    Surname,
    Email,
    Gender,
    Age,
    /*age groups*/
    CASE
        WHEN Age BETWEEN 0 AND 9 THEN '0-9'
        WHEN Age BETWEEN 10 AND 19 THEN '10-19'
        WHEN Age BETWEEN 20 AND 29 THEN '20-29'
        WHEN Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '70+'
    END AS Age_Group,
    /*Age categories*/
    CASE 
        WHEN Age BETWEEN 0 AND 12 THEN 'Child'
        WHEN Age BETWEEN 13 AND 17 THEN 'Teen'
        WHEN Age BETWEEN 18 AND 24 THEN 'Young Adult'
        WHEN Age BETWEEN 25 AND 34 THEN 'Adult'
        WHEN Age BETWEEN 35 AND 44 THEN 'Mid Adult'
        WHEN Age BETWEEN 45 AND 54 THEN 'Mature Adult'
        ELSE 'Senior'
    END AS Age_Category,
    Race,
    Province,
    Instagram_Handles,
    Channel2,
    /*Channel Categories*/
    CASE
        WHEN Channel2 IN ('Supersport Live Events','ICC Cricket World Cup 2011','SuperSport Blitz','SuperSport Live Events','Wimbledon','Live on SuperSport')
            THEN 'Sports'
        WHEN Channel2 IN ('Trace TV','Channel O','E! Entertainment','Vuzu','MK')
            THEN 'Music & Entertainment'
        WHEN Channel2 IN ('Africa Magic','M-Net','kykNET','Sawsee')
            THEN 'Movies & Series'
        WHEN Channel2 IN ('Cartoon Network','Boomerang')
            THEN 'Kids'
        ELSE 'Events & Technical'
    END AS Channel_Category,
    Number_Of_Views,
    LocalTime_SAST,
    /*time category*/
    CASE
        WHEN hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 0 AND 3 THEN 'Mid-Night to Early Morning'
        WHEN hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 4 AND 7 THEN 'Morning'
        WHEN hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 8 AND 11 THEN 'Late Morning'
        WHEN hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 12 AND 15 THEN 'Afternoon'
        WHEN hour(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm')) BETWEEN 16 AND 19 THEN 'Evening'
        ELSE 'Night'
    END AS Time_Category,
    /*day of the week by name EEEE*/
    date_format(to_timestamp(LocalTime_SAST, 'M/d/yyyy H:mm'),'EEEE') AS Day_Of_Week,
    /*Screen time per subscribed viewer*/
    CONCAT(
        LPAD(CAST(FLOOR(max_seconds / 3600) AS STRING), 2, '0'), ':',
        LPAD(CAST(FLOOR((max_seconds % 3600) / 60) AS STRING), 2, '0'), ':',
        LPAD(CAST(FLOOR(max_seconds % 60) AS STRING), 2, '0')) 
        AS Screen_time
FROM duration_calc;